# Qwen3-TTS Voice Clone Studio · Colab edition

1. Open this notebook in [Google Colab](https://colab.research.google.com/).
2. Go to **Runtime → Change runtime type** and pick **GPU**.
3. Run the launch cell below. It installs everything, loads the models, and opens the app.
4. Wait for **✅ READY** before you start.
5. Upload a recording, choose the parts of the audio to use, add or auto-transcribe the matching text, then generate.

**First run:** downloading the models can take several minutes. The app appears once both models (voice and transcription) have loaded. Running the cell again with the same settings keeps your session; changing models or updating the code restarts the app. Downloaded models stay cached until the runtime ends.

**Updating from an older version:** choose **Runtime → Restart session**, then run the launch cell. Your installed packages and downloaded models are kept.

**Default settings:** 1.7B voice model, Whisper base for transcription (runs on CPU), English, and mono 16-bit WAV output. Choose the **0.6B model on a T4 GPU** or if you run low on GPU memory. Older GPUs use more memory because they can't run the model in its faster BF16 format; newer GPUs (Ampere and later) can.

**Converting audio (optional):** after generating, turn on **Optional WAV conversion**, pick your settings, and click **Create WAV copy**. The default is mono, 8 kHz, 16-bit WAV. Preview it, then click **Download converted WAV**. The regular **Download WAV** and history ZIP always contain the original files. Note that converting to higher quality can't add detail that wasn't in the original.

The app runs directly inside the notebook, so you don't need a separate server.

**Before you finish:** download your files before the runtime ends. Generated audio is saved in `/content/qwen3_tts_outputs/` and reference files in `/content/qwen3_tts_work/`. Only clone voices you have permission to use.

In [ ]:
# @title ▶ Launch Voice Clone Studio (run this cell only)
MODEL_SIZE = "1.7B"  # @param ["1.7B", "0.6B"]
WHISPER_SIZE = "base"  # @param ["tiny", "base", "small"]
STUDIO_VERSION = "4.2.0"

import importlib
import importlib.metadata
import importlib.util
import subprocess
import sys
import tempfile
from pathlib import Path

try:
    from google.colab import output
except ImportError as error:
    raise RuntimeError("Open this notebook in Google Colab and run the launch cell.") from error


def ensure_packages(packages):
    """Skip pip when installed; preserve the runtime's numerical and GPU stack."""
    if getattr(sys, "_qwen_studio_restart_required", False):
        raise RuntimeError("Restart the Colab session after dependency updates, then rerun the launch cell.")
    missing = []
    for name, wanted in packages.items():
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        if installed is None or (wanted and installed != wanted):
            missing.append(f"{name}=={wanted}" if wanted else name)
    if not missing:
        return
    print("Installing: " + ", ".join(missing), flush=True)
    loaded_versions = {}
    for module, distributions in importlib.metadata.packages_distributions().items():
        if module in sys.modules:
            for name in distributions:
                try:
                    loaded_versions[name] = importlib.metadata.version(name)
                except importlib.metadata.PackageNotFoundError:
                    pass
    protected = []
    for name in ("numpy", "scipy", "torch", "torchaudio", "torchvision", "scikit-learn", "numba", "llvmlite"):
        try:
            protected.append(f"{name}=={importlib.metadata.version(name)}")
        except importlib.metadata.PackageNotFoundError:
            pass
    with tempfile.TemporaryDirectory() as directory:
        constraints = Path(directory) / "constraints.txt"
        constraints.write_text("\n".join(protected))
        result = subprocess.run([
            sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
            "--upgrade-strategy", "only-if-needed", "--constraint", str(constraints),
            *(f"{name}=={wanted}" if wanted else name for name, wanted in packages.items()),
        ], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(
            "Dependency installation failed. " + result.stderr[-2500:] +
            "\nIf this runtime contains conflicting packages, use Runtime → Disconnect and delete runtime, then rerun."
        )
    importlib.invalidate_caches()
    changed = [name for name, version in loaded_versions.items()
               if importlib.metadata.version(name) != version]
    if changed:
        sys._qwen_studio_restart_required = True
        raise RuntimeError(
            "Setup updated already-imported packages: " + ", ".join(changed) +
            ". Restart the Colab session (Runtime → Restart session), then run the launch cell again."
        )


def load_studio_models(model_size, whisper_size):
    """Install one compatible dependency set before importing either model library.

    Keep startup outside widgets.Output: in IPython its context manager can
    suppress exceptions, which previously allowed a false model-ready message.
    """
    import gc
    import os
    import shutil
    import torch

    loaded = {}
    print("Starting Voice Clone Studio. Please wait until the READY message.", flush=True)
    try:
        if not torch.cuda.is_available():
            raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun the launch cell.")
        print("[1/4] Installing/checking all dependencies together…", flush=True)
        ensure_packages({
            "ipywidgets": None, "soundfile": None, "scipy": None,
            "qwen-tts": "0.1.1", "transformers": "4.57.3", "accelerate": "1.12.0",
            "faster-whisper": "1.2.1",
        })
        if shutil.which("ffmpeg") is None:
            print("Installing FFmpeg…", flush=True)
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
            if shutil.which("ffmpeg") is None:
                raise RuntimeError("FFmpeg installation did not provide the ffmpeg executable.")

        # Both packages are installed before either import can load hub/tokenizers.
        from qwen_tts import Qwen3TTSModel
        from faster_whisper import WhisperModel

        major, _ = torch.cuda.get_device_capability(0)
        dtype = torch.bfloat16 if major >= 8 and torch.cuda.is_bf16_supported() else torch.float32
        torch.backends.cuda.matmul.allow_tf32 = major >= 8
        if dtype == torch.float32:
            print("Using float32 for stability on this GPU. Select 0.6B if memory is limited.", flush=True)
        model_id = f"Qwen/Qwen3-TTS-12Hz-{model_size}-Base"
        print(f"[2/4] Downloading/loading voice model: {model_id} ({dtype})…", flush=True)
        loaded["tts"] = Qwen3TTSModel.from_pretrained(
            model_id, device_map="cuda:0", dtype=dtype,
            attn_implementation="sdpa", low_cpu_mem_usage=True,
        )
        if loaded["tts"] is None:
            raise RuntimeError("The voice model loader returned no model.")

        print(f"[3/4] Downloading/loading Whisper {whisper_size} on CPU…", flush=True)
        loaded["whisper"] = WhisperModel(
            whisper_size, device="cpu", compute_type="int8",
            cpu_threads=max(1, min(4, os.cpu_count() or 2)), num_workers=1,
        )
        if loaded["whisper"] is None:
            raise RuntimeError("The transcription model loader returned no model.")
        return loaded
    except Exception as error:
        # Do not retain the voice model if Whisper or another startup step fails.
        loaded.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"STARTUP FAILED — the studio is not ready: {error}", flush=True)
        if isinstance(error, torch.cuda.OutOfMemoryError):
            print("Select MODEL_SIZE='0.6B' and rerun, or use a GPU with more memory.", flush=True)
        raise


def create_studio(model_size, whisper_size, models):
    from datetime import datetime
    from pathlib import Path
    from string import Template
    import base64
    import gc
    import html
    import json
    import math
    import os
    import random
    import re
    import shutil
    import subprocess
    import uuid
    import zipfile

    import numpy as np
    import soundfile as sf
    import torch
    import ipywidgets as widgets

    from IPython.display import Audio, HTML, JSON, clear_output, display
    from google.colab import files, output


    # =========================================================
    # Runtime checks / configuration
    # =========================================================

    STUDIO_MODEL_ID = f"Qwen/Qwen3-TTS-12Hz-{model_size}-Base"
    STUDIO_GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU needed for generation"
    STUDIO_MIN_REFERENCE_SECONDS = 1.0
    STUDIO_RECOMMENDED_REFERENCE_SECONDS = (5.0, 15.0)
    STUDIO_MAX_REFERENCE_SECONDS = 20.0
    STUDIO_MAX_SCRIPT_CHARS = 8000
    STUDIO_NORMALIZE_SR = 24000
    STUDIO_MAX_REGIONS = 16
    STUDIO_MAX_UPLOAD_MB = 100
    STUDIO_MAX_SOURCE_SECONDS = 600.0
    STUDIO_WORK_DIR = Path("/content/qwen3_tts_work")
    STUDIO_OUTPUT_DIR = Path("/content/qwen3_tts_outputs")
    STUDIO_WORK_DIR.mkdir(parents=True, exist_ok=True)
    STUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ALLOWED_AUDIO_EXTENSIONS = {
        ".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".opus", ".webm"
    }

    # Pin browser UI dependency so CDN updates cannot silently alter the editor.
    WAVESURFER_VERSION = "7.12.6"

    output.enable_custom_widget_manager()



    # The interface is constructed only after both startup model loads succeed.
    if any(models.get(name) is None for name in ("tts", "whisper")):
        raise RuntimeError("Both models must be loaded before opening the studio. Rerun the launch cell.")
    model_status = widgets.HTML("<small>Voice model: ready · Transcription: ready (CPU)</small>")


    def update_model_status():
        voice = "ready" if models["tts"] is not None else "not loaded — rerun launch cell"
        transcription = "ready (CPU)" if models["whisper"] is not None else "not loaded — rerun launch cell"
        model_status.value = f"<small>Voice model: {voice} · Transcription: {transcription}</small>"


    def ensure_ffmpeg():
        ffmpeg = shutil.which("ffmpeg")
        if ffmpeg is None:
            raise RuntimeError("FFmpeg is unavailable. Rerun the launch cell to repair setup.")
        return ffmpeg


    def get_tts_model():
        model = models.get("tts")
        if model is None:
            raise RuntimeError("Voice model is unavailable. Rerun the launch cell before generating.")
        return model


    def get_whisper_model():
        model = models.get("whisper")
        if model is None:
            raise RuntimeError("Transcription model is unavailable. Rerun the launch cell before transcribing.")
        return model


    # =========================================================
    # Small self-contained audio helpers
    # =========================================================


    def _safe_filename(name, default="audio.wav"):
        name = Path(str(name or default)).name
        stem = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("._")
        return stem or default


    def seconds_to_timestamp(seconds):
        seconds = max(0.0, float(seconds))
        minutes = int(seconds // 60)
        remainder = seconds - minutes * 60
        return f"{minutes:02d}:{remainder:04.1f}"


    def audio_metrics_from_array(audio, sr):
        audio = np.asarray(audio, dtype=np.float32).reshape(-1)
        if audio.size == 0:
            return {"duration": 0.0, "peak": 0.0, "rms_dbfs": -120.0}
        peak = float(np.max(np.abs(audio)))
        rms = float(np.sqrt(np.mean(np.square(audio), dtype=np.float64)))
        rms_dbfs = 20.0 * math.log10(max(rms, 1e-6))
        return {
            "duration": float(len(audio) / int(sr)),
            "peak": peak,
            "rms_dbfs": rms_dbfs,
        }


    def load_mono_audio(path):
        audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
        audio = np.asarray(audio, dtype=np.float32)
        if audio.ndim == 2:
            audio = np.mean(audio, axis=1, dtype=np.float32)
        audio = np.ascontiguousarray(audio.reshape(-1), dtype=np.float32)
        if audio.size == 0 or int(sr) <= 0:
            raise ValueError("Audio is empty or has an invalid sample rate.")
        if not np.isfinite(audio).all():
            audio = np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0)
        return audio, int(sr)


    def load_whisper_audio(path):
        """Whisper ndarray input must be mono float32 at 16 kHz, not the 24 kHz reference rate.

        Decode with SoundFile and resample here so faster-whisper never calls
        av.open(metadata_errors=...), which is incompatible with PyAV 19.
        """
        from scipy.signal import resample_poly
        audio, sr = load_mono_audio(path)
        if sr != 16000:
            divisor = math.gcd(sr, 16000)
            audio = resample_poly(audio, 16000 // divisor, sr // divisor)
        return np.ascontiguousarray(audio, dtype=np.float32)


    def normalize_uploaded_audio_v3(name, content):
        """Save browser bytes and normalize to 24 kHz mono PCM WAV using FFmpeg."""
        if not content:
            raise ValueError("The uploaded file is empty.")

        safe_name = _safe_filename(name, "reference_audio")
        suffix = Path(safe_name).suffix.lower() or ".bin"
        token = uuid.uuid4().hex[:10]
        raw_path = STUDIO_WORK_DIR / f"upload_{token}{suffix}"
        normalized_path = STUDIO_WORK_DIR / f"reference_{token}.wav"
        raw_path.write_bytes(bytes(content))

        try:
            ffmpeg = ensure_ffmpeg()
            if ffmpeg:
                command = [
                    ffmpeg,
                    "-hide_banner",
                    "-loglevel",
                    "error",
                    "-y",
                    "-nostdin",
                    "-i",
                    str(raw_path),
                    "-t",
                    str(STUDIO_MAX_SOURCE_SECONDS + 1),
                    "-vn",
                    "-ac",
                    "1",
                    "-ar",
                    str(STUDIO_NORMALIZE_SR),
                    "-c:a",
                    "pcm_s16le",
                    str(normalized_path),
                ]
                try:
                    completed = subprocess.run(
                        command,
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE,
                        text=True,
                        timeout=120,
                    )
                except subprocess.TimeoutExpired as exc:
                    raise RuntimeError("FFmpeg timed out while decoding the upload.") from exc
                if completed.returncode != 0:
                    detail = completed.stderr.strip()[-1200:]
                    raise RuntimeError(f"FFmpeg could not decode this audio file. {detail}")
            else:
                # Fallback for formats directly supported by libsndfile.
                audio, sr = sf.read(str(raw_path), dtype="float32", always_2d=False)
                audio = np.asarray(audio, dtype=np.float32)
                if audio.ndim == 2:
                    audio = np.mean(audio, axis=1, dtype=np.float32)
                if int(sr) != STUDIO_NORMALIZE_SR:
                    raise RuntimeError(
                        "FFmpeg is unavailable and this file needs resampling. "
                        "Colab normally includes FFmpeg; reconnect the runtime if it is missing."
                    )
                sf.write(str(normalized_path), audio, STUDIO_NORMALIZE_SR, subtype="PCM_16")
        except Exception:
            normalized_path.unlink(missing_ok=True)
            raise
        finally:
            try:
                raw_path.unlink(missing_ok=True)
            except Exception:
                pass

        info = sf.info(str(normalized_path))
        duration = float(info.frames / info.samplerate) if info.samplerate else 0.0
        if duration <= 0:
            normalized_path.unlink(missing_ok=True)
            raise ValueError("No decodable audio was found in the uploaded file.")
        return str(normalized_path), duration


    def next_available_path(directory, filename):
        directory = Path(directory)
        directory.mkdir(parents=True, exist_ok=True)
        filename = _safe_filename(filename, "cloned_voice.wav")
        path = directory / filename
        if not path.suffix:
            path = path.with_suffix(".wav")
        if not path.exists():
            return str(path)
        stem = path.stem
        suffix = path.suffix
        for index in range(2, 10000):
            candidate = directory / f"{stem}_{index}{suffix}"
            if not candidate.exists():
                return str(candidate)
        raise RuntimeError("Could not allocate a unique output filename.")


    def sanitize_generated_audio(audio):
        audio = np.asarray(audio, dtype=np.float32).squeeze()
        if audio.ndim == 0:
            audio = audio.reshape(1)
        if audio.ndim != 1 or not audio.size:
            raise ValueError("The model returned empty or non-mono audio. No output was saved.")
        if not np.isfinite(audio).all():
            raise ValueError("The model returned NaN/Inf audio. Restart the runtime and try the 0.6B model.")
        return np.ascontiguousarray(audio, dtype=np.float32)


    def split_text(text, max_chars):
        """Punctuation-aware chunking for long TTS scripts."""
        text = re.sub(r"\s+", " ", str(text or "")).strip()
        if not text:
            return []
        max_chars = max(40, int(max_chars))
        if len(text) <= max_chars:
            return [text]

        sentences = re.split(r"(?<=[.!?。！？；;:])\s+", text)
        chunks = []
        current = ""

        def flush():
            nonlocal current
            if current.strip():
                chunks.append(current.strip())
            current = ""

        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            if len(sentence) <= max_chars:
                candidate = sentence if not current else current + " " + sentence
                if len(candidate) <= max_chars:
                    current = candidate
                else:
                    flush()
                    current = sentence
                continue

            flush()
            words = sentence.split(" ")
            piece = ""
            for word in words:
                if len(word) > max_chars:
                    if piece:
                        chunks.append(piece)
                        piece = ""
                    for start in range(0, len(word), max_chars):
                        chunks.append(word[start : start + max_chars])
                    continue
                candidate = word if not piece else piece + " " + word
                if len(candidate) <= max_chars:
                    piece = candidate
                else:
                    chunks.append(piece)
                    piece = word
            if piece:
                chunks.append(piece)
        flush()
        return chunks


    def join_generated_audio(parts, sr, gap_seconds=0.22, fade_ms=8, peak_normalize=True):
        cleaned = [sanitize_generated_audio(part) for part in parts]
        if not cleaned:
            raise ValueError("No generated audio sections are available.")

        sr = int(sr)
        gap = np.zeros(max(0, int(round(float(gap_seconds) * sr))), dtype=np.float32)
        fade_samples = max(0, int(round(float(fade_ms) * sr / 1000.0)))
        assembled = []

        for index, part in enumerate(cleaned):
            part = part.copy()
            n = min(fade_samples, len(part) // 2)
            if n > 0:
                ramp = np.linspace(0.0, 1.0, n, endpoint=True, dtype=np.float32)
                part[:n] *= ramp
                part[-n:] *= ramp[::-1]
            if index and gap.size:
                assembled.append(gap)
            assembled.append(part)

        result = np.concatenate(assembled).astype(np.float32, copy=False)
        if peak_normalize and result.size:
            peak = float(np.max(np.abs(result)))
            target = float(10 ** (-1.0 / 20.0))
            if peak > 1e-7:
                result = result * (target / peak)
        return np.clip(result, -1.0, 1.0).astype(np.float32, copy=False)


    def file_to_data_url(path, mime="audio/wav"):
        encoded = base64.b64encode(Path(path).read_bytes()).decode("ascii")
        return f"data:{mime};base64,{encoded}"


    # =========================================================
    # State
    # =========================================================

    state = {
        "editor_token": None,
        "editor_revision": -1,
        "original_path": None,
        "original_name": None,
        "duration": 0.0,
        "segments": [],
        "selected_region_id": None,
        "segments_signature": None,
        "stitched_path": None,
        "stitched_duration": 0.0,
        "stitched_speech_duration": 0.0,
        "stitched_signature": None,
        "voice_prompt": None,
        "voice_prompt_key": None,
        "generated_path": None,
        "generated_sr": None,
        "history": [],
    }

    CALLBACK_PREFIX = f"qwen3_studio_v3_{uuid.uuid4().hex[:10]}"
    SEGMENTS_CALLBACK = CALLBACK_PREFIX + ".segments"


    def invalidate_prompt_cache():
        state["voice_prompt"] = None
        state["voice_prompt_key"] = None


    def ordered_segments():
        duration = float(state.get("duration") or 0.0)
        cleaned = []
        for index, segment in enumerate(state.get("segments") or []):
            try:
                start = max(0.0, min(float(segment["start"]), duration))
                end = max(start, min(float(segment["end"]), duration))
            except Exception:
                continue
            if end - start < 0.05:
                continue
            cleaned.append(
                {
                    "id": str(segment.get("id") or f"segment_{index + 1}"),
                    "start": start,
                    "end": end,
                    "order": int(segment.get("order", index)),
                }
            )
        cleaned.sort(key=lambda item: (item["order"], item["start"], item["end"]))
        return cleaned


    def segment_signature(segments=None):
        segments = ordered_segments() if segments is None else segments
        return tuple(
            (item["id"], round(item["start"], 3), round(item["end"], 3), int(item["order"]))
            for item in segments
        )


    def selected_speech_duration():
        return sum(item["end"] - item["start"] for item in ordered_segments())


    # =========================================================
    # Modern UI helpers
    # =========================================================

    STYLE = widgets.HTML(
        """
        <style>
          .q3v3-shell { font-family: Inter, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }
          .q3v3-card {
            border: 1px solid rgba(127,127,127,.22) !important;
            border-radius: 18px !important;
            padding: 18px 18px 16px 18px !important;
            margin: 0 0 14px 0 !important;
            box-shadow: 0 8px 28px rgba(0,0,0,.045);
            background: rgba(127,127,127,.035);
          }
          .q3v3-card .widget-label { font-weight: 600; }
          .q3v3-card .widget-button {
            border-radius: 11px !important;
            font-weight: 600 !important;
            min-height: 36px;
          }
          .q3v3-card textarea, .q3v3-card input, .q3v3-card select {
            border-radius: 10px !important;
          }
          .q3v3-card .widget-textarea textarea {
            line-height: 1.5 !important;
          }
          .q3v3-tabs .p-TabBar-tab {
            border-radius: 11px 11px 0 0 !important;
            font-weight: 600;
          }
        </style>
        """
    )


    def card(*children):
        box = widgets.VBox(
            list(children),
            layout=widgets.Layout(width="100%"),
        )
        box.add_class("q3v3-card")
        return box


    def section_heading(icon, title, subtitle=""):
        subtitle_html = (
            f"<div style='opacity:.66;font-size:12.5px;line-height:1.45;margin-top:4px'>{subtitle}</div>"
            if subtitle
            else ""
        )
        return widgets.HTML(
            f"""
            <div style="margin:0 0 13px 0">
              <div style="display:flex;align-items:center;gap:10px;font-size:17px;font-weight:750;line-height:1.3">
                <span style="width:32px;height:32px;border-radius:10px;display:inline-flex;align-items:center;
                             justify-content:center;background:rgba(26,115,232,.12);font-size:16px">{icon}</span>
                <span>{html.escape(title)}</span>
              </div>
              {subtitle_html}
            </div>
            """
        )


    def set_status(message="", kind="info"):
        palette = {
            "info": ("rgba(26,115,232,.10)", "#1967d2", "●"),
            "success": ("rgba(24,128,56,.10)", "#188038", "●"),
            "warning": ("rgba(249,171,0,.12)", "#b06000", "●"),
            "error": ("rgba(217,48,37,.10)", "#c5221f", "●"),
        }
        if not message:
            status_banner.value = ""
            return
        bg, fg, dot = palette.get(kind, palette["info"])
        status_banner.value = (
            f"<div style='background:{bg};color:{fg};border:1px solid {fg}22;border-radius:13px;"
            f"padding:10px 13px;margin:2px 0 12px;font-size:13px;line-height:1.45'>"
            f"<span style='margin-right:7px'>{dot}</span>{message}</div>"
        )


    def log(message):
        stamp = datetime.now().strftime("%H:%M:%S")
        with log_output:
            print(f"[{stamp}] {message}")


    def set_busy(widget, busy, busy_description=None):
        if not hasattr(widget, "_q3v3_description"):
            widget._q3v3_description = getattr(widget, "description", "")
            widget._q3v3_icon = getattr(widget, "icon", "")
        widget.disabled = bool(busy)
        if hasattr(widget, "description"):
            widget.description = (
                busy_description if busy and busy_description else widget._q3v3_description
            )
        if hasattr(widget, "icon"):
            widget.icon = "spinner" if busy else widget._q3v3_icon


    # =========================================================
    # Header / status / logs
    # =========================================================

    header = widgets.HTML(
        f"""
        <div class="q3v3-shell" style="padding:4px 2px 16px">
          <div style="display:flex;align-items:flex-start;justify-content:space-between;gap:16px;flex-wrap:wrap">
            <div>
              <div style="font-size:27px;font-weight:780;letter-spacing:-.35px;line-height:1.2">
                Qwen3-TTS Voice Clone Studio <span style="opacity:.42;font-weight:650">v{html.escape(STUDIO_VERSION)}</span>
              </div>
              <div style="font-size:13px;opacity:.68;margin-top:6px">
                Multi-splice waveform editor · native Colab upload · reusable clone prompts
              </div>
            </div>
            <div style="display:flex;gap:7px;flex-wrap:wrap;justify-content:flex-end">
              <span style="padding:6px 9px;border-radius:999px;background:rgba(26,115,232,.10);font-size:11.5px;font-weight:650">{html.escape(str(STUDIO_GPU_NAME))}</span>
              <span style="padding:6px 9px;border-radius:999px;background:rgba(127,127,127,.10);font-size:11.5px;font-weight:650">WaveSurfer {WAVESURFER_VERSION}</span>
              <span style="padding:6px 9px;border-radius:999px;background:rgba(24,128,56,.10);font-size:11.5px;font-weight:650">Colab native</span>
            </div>
          </div>
          <div style="font-size:11.5px;opacity:.50;margin-top:8px">{html.escape(str(STUDIO_MODEL_ID))} · outputs: {html.escape(str(STUDIO_OUTPUT_DIR))}</div>
        </div>
        """
    )

    status_banner = widgets.HTML()
    log_output = widgets.Output(
        layout=widgets.Layout(max_height="230px", overflow="auto", width="100%")
    )
    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=1,
        description="",
        bar_style="info",
        layout=widgets.Layout(width="100%", display="none"),
    )


    # =========================================================
    # Reference upload + WaveSurfer editor
    # =========================================================

    reference_upload = widgets.Button(
        description="Upload reference",
        icon="upload",
        button_style="info",
        layout=widgets.Layout(width="180px", height="40px"),
    )
    reference_info = widgets.HTML(
        "<span style='opacity:.62'>No reference audio loaded.</span>"
    )
    upload_transport = widgets.Output(layout=widgets.Layout(width="100%"))
    waveform_output = widgets.Output(layout=widgets.Layout(width="100%"))
    segment_summary = widgets.HTML(
        "<span style='opacity:.62'>Upload audio, then drag across the waveform to create splice regions.</span>"
    )

    splice_gap_ms = widgets.IntSlider(
        description="Join gap",
        value=35,
        min=0,
        max=250,
        step=5,
        continuous_update=False,
        readout=True,
        readout_format="d",
        style={"description_width": "72px"},
        layout=widgets.Layout(width="390px"),
    )
    splice_fade_ms = widgets.IntSlider(
        description="Edge fade",
        value=5,
        min=0,
        max=30,
        step=1,
        continuous_update=False,
        readout=True,
        readout_format="d",
        style={"description_width": "72px"},
        layout=widgets.Layout(width="390px"),
    )

    preview_splice_button = widgets.Button(
        description="Preview splice",
        icon="play",
        disabled=True,
        layout=widgets.Layout(width="150px"),
    )
    download_splice_button = widgets.Button(
        description="Download splice",
        icon="download",
        disabled=True,
        layout=widgets.Layout(width="155px"),
    )
    rerender_editor_button = widgets.Button(
        description="Reload waveform",
        icon="refresh",
        disabled=True,
        layout=widgets.Layout(width="155px"),
    )
    splice_player = widgets.Output(layout=widgets.Layout(width="100%"))

    editor_settings = widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-size:12.5px;font-weight:700;margin:10px 0 4px'>Splice assembly settings</div>"
                "<div style='font-size:12px;opacity:.68;margin-bottom:7px'>"
                "A tiny gap/fade prevents clicks where non-adjacent clips are joined. "
                "These settings affect the assembled reference sent to Whisper/Qwen, not the source file.</div>"
            ),
            splice_gap_ms,
            splice_fade_ms,
        ],
        layout=widgets.Layout(width="100%"),
    )


    def update_segment_summary():
        segments = ordered_segments()
        count = len(segments)
        speech_duration = sum(item["end"] - item["start"] for item in segments)
        low, high = STUDIO_RECOMMENDED_REFERENCE_SECONDS

        if count == 0:
            segment_summary.value = (
                "<div style='font-size:12.5px;opacity:.65'>"
                "No splice regions yet. <b>Drag across the waveform</b> to add one."
                "</div>"
            )
            preview_splice_button.disabled = True
            download_splice_button.disabled = True
            return

        if speech_duration > STUDIO_MAX_REFERENCE_SECONDS:
            badge = (
                f"<span style='color:#c5221f;font-weight:700'>"
                f"over {STUDIO_MAX_REFERENCE_SECONDS:g}s clone limit</span>"
            )
        elif low <= speech_duration <= high:
            badge = "<span style='color:#188038;font-weight:700'>ideal clone range</span>"
        elif speech_duration < STUDIO_MIN_REFERENCE_SECONDS:
            badge = "<span style='color:#c5221f;font-weight:700'>too short</span>"
        else:
            badge = "<span style='opacity:.65;font-weight:650'>usable</span>"

        segment_summary.value = (
            f"<div style='display:flex;gap:9px;align-items:center;flex-wrap:wrap;font-size:12.5px'>"
            f"<b>{count} region{'s' if count != 1 else ''}</b>"
            f"<span style='opacity:.38'>•</span>"
            f"<span>{speech_duration:.2f}s selected / {STUDIO_MAX_REFERENCE_SECONDS:g}s max</span>"
            f"<span style='opacity:.38'>•</span>{badge}</div>"
        )
        preview_splice_button.disabled = False
        download_splice_button.disabled = False


    def _mark_splice_dirty(*_):
        state["stitched_path"] = None
        state["stitched_duration"] = 0.0
        state["stitched_speech_duration"] = 0.0
        invalidate_prompt_cache()


    splice_gap_ms.observe(_mark_splice_dirty, names="value")
    splice_fade_ms.observe(_mark_splice_dirty, names="value")


    WAVEFORM_TEMPLATE = Template(
        r"""
    <div id="$root_id" class="q3-wave-root" tabindex="0">
      <style>
        #$root_id {
          --q3-accent:#1a73e8;
          --q3-accent2:#6f5cff;
          --q3-text:inherit;
          border:1px solid rgba(127,127,127,.22);
          border-radius:18px;
          padding:14px;
          margin:2px 0 10px;
          background:linear-gradient(180deg, rgba(127,127,127,.035), rgba(127,127,127,.015));
          outline:none;
          font-family:Inter,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;
        }
        #$root_id .q3-topbar {
          display:flex;align-items:center;justify-content:space-between;gap:10px;flex-wrap:wrap;margin-bottom:10px;
        }
        #$root_id .q3-controls {display:flex;gap:7px;align-items:center;flex-wrap:wrap}
        #$root_id button {
          border:1px solid rgba(127,127,127,.24);background:rgba(127,127,127,.06);color:inherit;
          min-height:34px;border-radius:10px;padding:6px 10px;font:600 12px/1.1 inherit;cursor:pointer;
        }
        #$root_id button:hover {background:rgba(26,115,232,.10);border-color:rgba(26,115,232,.30)}
        #$root_id button.primary {background:rgba(26,115,232,.12);border-color:rgba(26,115,232,.26);color:#1967d2}
        #$root_id .q3-time {font:650 12px/1.1 ui-monospace,SFMono-Regular,Menlo,monospace;opacity:.68}
        #$root_id .q3-wave-shell {
          border:1px solid rgba(127,127,127,.17);border-radius:14px;padding:10px 10px 5px;background:rgba(127,127,127,.025);
        }
        #$root_id #$wave_id {min-height:170px;cursor:pointer}
        #$root_id #$wave_id ::part(cursor) {width:2px;background:var(--q3-accent)}
        #$root_id #$wave_id ::part(region) {border-radius:7px}
        #$root_id .q3-help {font-size:11.5px;opacity:.58;margin:8px 2px 2px;line-height:1.45}
        #$root_id .q3-list {display:flex;flex-direction:column;gap:6px;margin-top:11px}
        #$root_id .q3-row {
          display:grid;grid-template-columns:30px minmax(80px,1fr) auto;gap:8px;align-items:center;
          padding:7px 8px;border:1px solid rgba(127,127,127,.16);border-radius:11px;background:rgba(127,127,127,.028);
        }
        #$root_id .q3-index {
          width:27px;height:27px;border-radius:8px;background:rgba(26,115,232,.11);display:flex;align-items:center;
          justify-content:center;color:#1967d2;font-weight:750;font-size:11px;
        }
        #$root_id .q3-range {font:600 11.5px/1.2 ui-monospace,SFMono-Regular,Menlo,monospace;white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
        #$root_id .q3-row-actions {display:flex;gap:4px}
        #$root_id .q3-row-actions button {min-height:29px;padding:4px 7px;border-radius:8px;font-size:11px}
        #$root_id .q3-empty {font-size:12px;opacity:.55;padding:9px 3px}
        #$root_id .q3-status {font-size:11.5px;opacity:.65;margin-top:8px;min-height:16px}
        #$root_id .q3-zoom {display:flex;gap:6px;align-items:center;font-size:11.5px;opacity:.70}
        #$root_id .q3-zoom input {width:110px}
        #$root_id audio {width:100%;margin-top:8px;display:none}
        @media (max-width:700px){
          #$root_id .q3-row {grid-template-columns:30px 1fr}
          #$root_id .q3-row-actions {grid-column:2;justify-content:flex-start}
        }
      </style>

      <div class="q3-topbar">
        <div class="q3-controls">
          <button class="primary" id="$play_id">▶ Play</button>
          <button id="$play_splice_id">▶ Splice</button>
          <button id="$add_id">＋ 3s at playhead</button>
          <button id="$clear_id">Clear regions</button>
        </div>
        <div style="display:flex;gap:12px;align-items:center;flex-wrap:wrap">
          <div class="q3-zoom"><span>Zoom</span><input id="$zoom_id" type="range" min="10" max="220" value="45"></div>
          <div id="$time_id" class="q3-time">00:00.0 / 00:00.0</div>
        </div>
      </div>

      <div class="q3-wave-shell">
        <div id="$wave_id"></div>
      </div>

      <div class="q3-help">
        <b>Drag</b> across empty waveform space to add a splice region. Drag a region to move it; drag its edges to resize it.
        Click a region to play that section. Click the waveform itself to seek/play. Use ↑/↓ below to change splice order.
      </div>
      <div id="$list_id" class="q3-list"></div>
      <div id="$status_id" class="q3-status">Loading waveform…</div>
      <audio id="$audio_id" controls preload="metadata" src="$audio_url"></audio>

      <script type="module">
      (async () => {
        const root = document.getElementById('$root_id');
        const statusEl = document.getElementById('$status_id');
        const listEl = document.getElementById('$list_id');
        const timeEl = document.getElementById('$time_id');
        const playButton = document.getElementById('$play_id');
        const spliceButton = document.getElementById('$play_splice_id');
        const addButton = document.getElementById('$add_id');
        const clearButton = document.getElementById('$clear_id');
        const zoomInput = document.getElementById('$zoom_id');

        const callbackName = '$callback_name';
        const initialSegments = $segments_json;
        const editorToken = '$editor_token';
        let revision = 0;
        let syncChain = Promise.resolve();
        const maxRegions = $max_regions;
        const durationHint = $duration;

        let ws;
        let regions;
        let selectedId = null;
        let orderMap = new Map();
        let syncTimer = null;
        let stopAt = null;
        let playingSplice = false;
        let spliceQueue = [];
        let spliceIndex = -1;

        const fmt = (seconds) => {
          seconds = Math.max(0, Number(seconds || 0));
          const m = Math.floor(seconds / 60);
          const s = seconds - m * 60;
          return `$${String(m).padStart(2,'0')}:$${s.toFixed(1).padStart(4,'0')}`;
        };

        const setStatus = (text, isError=false) => {
          statusEl.textContent = text || '';
          statusEl.style.color = isError ? '#c5221f' : 'inherit';
          statusEl.style.opacity = isError ? '1' : '.65';
        };

        function sortedRegions() {
          if (!regions) return [];
          return regions.getRegions().slice().sort((a,b) => {
            const ao = orderMap.has(a.id) ? orderMap.get(a.id) : 9999;
            const bo = orderMap.has(b.id) ? orderMap.get(b.id) : 9999;
            return ao - bo || a.start - b.start;
          });
        }

        function normalizeOrder() {
          sortedRegions().forEach((region, index) => orderMap.set(region.id, index));
        }

        function payload() {
          normalizeOrder();
          return {
            editor_token: editorToken,
            revision: ++revision,
            selected_id: selectedId,
            segments: sortedRegions().map((r, index) => ({
              id: String(r.id),
              start: Number(r.start),
              end: Number(r.end),
              order: index,
            })),
          };
        }

        async function syncNow() {
          if (!root.isConnected) return;
          if (!(window.google && google.colab && google.colab.kernel)) {
            setStatus('Colab connection unavailable. Rerun the launch cell before using these regions.', true);
            return;
          }
          const data = payload();
          try {
            syncChain = syncChain.catch(() => {}).then(async () => {
              const result = await google.colab.kernel.invokeFunction(callbackName, [data], {});
              const reply = result?.data?.['application/json'];
              if (!reply?.ok) throw new Error(reply?.error || 'No region acknowledgement received');
            });
            await syncChain;
            if (data.revision === revision) setStatus('Regions synced. Ready to preview, transcribe, or generate.');
          } catch (error) {
            console.warn('Segment sync failed', error);
            setStatus('Could not sync splice regions to Python: ' + error, true);
          }
        }

        function scheduleSync() {
          setStatus('Syncing regions…');
          clearTimeout(syncTimer);
          syncTimer = setTimeout(syncNow, 90);
        }

        function playRange(region) {
          if (!region || !ws) return;
          playingSplice = false;
          stopAt = Number(region.end);
          selectedId = region.id;
          ws.setTime(Number(region.start));
          ws.play();
          renderList();
          scheduleSync();
        }

        function playSplice() {
          spliceQueue = sortedRegions();
          if (!spliceQueue.length) {
            setStatus('Add at least one region first.', true);
            return;
          }
          playingSplice = true;
          spliceIndex = 0;
          const region = spliceQueue[0];
          selectedId = region.id;
          stopAt = Number(region.end);
          ws.setTime(Number(region.start));
          ws.play();
          renderList();
          setStatus(`Playing splice: $${spliceQueue.length} region$${spliceQueue.length === 1 ? '' : 's'} in list order.`);
        }

        function advanceSplice() {
          spliceIndex += 1;
          if (!playingSplice || spliceIndex >= spliceQueue.length) {
            playingSplice = false;
            stopAt = null;
            ws.pause();
            setStatus('Splice preview finished.');
            return;
          }
          const region = spliceQueue[spliceIndex];
          selectedId = region.id;
          stopAt = Number(region.end);
          ws.setTime(Number(region.start));
          ws.play();
          renderList();
        }

        function renderList() {
          const items = sortedRegions();
          listEl.innerHTML = '';
          if (!items.length) {
            listEl.innerHTML = '<div class="q3-empty">No regions yet — drag across the waveform to create one.</div>';
            return;
          }

          items.forEach((region, index) => {
            const row = document.createElement('div');
            row.className = 'q3-row';
            if (region.id === selectedId) {
              row.style.borderColor = 'rgba(26,115,232,.38)';
              row.style.background = 'rgba(26,115,232,.055)';
            }

            const idx = document.createElement('div');
            idx.className = 'q3-index';
            idx.textContent = String(index + 1);

            const range = document.createElement('div');
            range.className = 'q3-range';
            range.textContent = `$${fmt(region.start)} → $${fmt(region.end)}  ·  $${(region.end-region.start).toFixed(2)}s`;

            const actions = document.createElement('div');
            actions.className = 'q3-row-actions';

            const play = document.createElement('button');
            play.textContent = '▶';
            play.title = 'Play this region';
            play.onclick = () => playRange(region);

            const up = document.createElement('button');
            up.textContent = '↑';
            up.title = 'Move earlier in splice';
            up.disabled = index === 0;
            up.onclick = () => {
              if (index <= 0) return;
              const previous = items[index - 1];
              orderMap.set(region.id, index - 1);
              orderMap.set(previous.id, index);
              normalizeOrder();
              renderList();
              scheduleSync();
            };

            const down = document.createElement('button');
            down.textContent = '↓';
            down.title = 'Move later in splice';
            down.disabled = index === items.length - 1;
            down.onclick = () => {
              if (index >= items.length - 1) return;
              const next = items[index + 1];
              orderMap.set(region.id, index + 1);
              orderMap.set(next.id, index);
              normalizeOrder();
              renderList();
              scheduleSync();
            };

            const remove = document.createElement('button');
            remove.textContent = '×';
            remove.title = 'Remove region';
            remove.onclick = () => region.remove();

            actions.append(play, up, down, remove);
            row.append(idx, range, actions);
            row.onclick = (event) => {
              if (event.target.tagName.toLowerCase() === 'button') return;
              selectedId = region.id;
              renderList();
              scheduleSync();
            };
            listEl.appendChild(row);
          });
        }

        try {
          const [{default: WaveSurfer}, {default: RegionsPlugin}] = await Promise.all([
            import('https://cdn.jsdelivr.net/npm/wavesurfer.js@$wavesurfer_version/dist/wavesurfer.esm.js'),
            import('https://cdn.jsdelivr.net/npm/wavesurfer.js@$wavesurfer_version/dist/plugins/regions.esm.js'),
          ]);

          regions = RegionsPlugin.create();
          ws = WaveSurfer.create({
            container: '#$wave_id',
            url: '$audio_url',
            waveColor: '#9aa0a6',
            progressColor: '#1a73e8',
            cursorColor: '#6f5cff',
            height: 170,
            barWidth: 2,
            barGap: 1,
            barRadius: 2,
            normalize: true,
            minPxPerSec: 45,
            autoCenter: true,
            plugins: [regions],
          });

          regions.enableDragSelection({
            color: 'rgba(26,115,232,.22)',
            drag: true,
            resize: true,
            minLength: 0.15,
          }, 4);

          ws.on('ready', (duration) => {
            initialSegments.forEach((segment, index) => {
              orderMap.set(String(segment.id), Number(segment.order ?? index));
              regions.addRegion({
                id: String(segment.id),
                start: Number(segment.start),
                end: Number(segment.end),
                color: index % 2 === 0 ? 'rgba(26,115,232,.22)' : 'rgba(111,92,255,.20)',
                drag: true,
                resize: true,
                minLength: 0.15,
              });
            });
            normalizeOrder();
            renderList();
            timeEl.textContent = `$${fmt(0)} / $${fmt(duration)}`;
            setStatus('Ready — drag to create regions; click waveform or a region to play.');
            scheduleSync();
          });

          ws.on('timeupdate', (time) => {
            timeEl.textContent = `$${fmt(time)} / $${fmt(ws.getDuration() || durationHint)}`;
            if (stopAt !== null && time >= stopAt - 0.018) {
              if (playingSplice) advanceSplice();
              else {
                stopAt = null;
                ws.pause();
              }
            }
          });

          ws.on('play', () => { playButton.textContent = '❚❚ Pause'; });
          ws.on('pause', () => { playButton.textContent = '▶ Play'; });
          ws.on('error', (error) => setStatus('Waveform audio error: ' + error, true));

          ws.on('click', () => {
            // WaveSurfer already seeks to the clicked position. Make the waveform itself a player.
            stopAt = null;
            playingSplice = false;
            ws.playPause();
          });

          regions.on('region-created', (region) => {
            if (!orderMap.has(region.id)) {
              if (regions.getRegions().length > maxRegions) {
                region.remove();
                setStatus(`Maximum $${maxRegions} regions per reference.`, true);
                return;
              }
              orderMap.set(region.id, sortedRegions().length - 1);
            }
            selectedId = region.id;
            normalizeOrder();
            renderList();
            scheduleSync();
          });

          regions.on('region-updated', (region) => {
            selectedId = region.id;
            renderList();
            scheduleSync();
          });

          regions.on('region-removed', (region) => {
            orderMap.delete(region.id);
            if (selectedId === region.id) selectedId = null;
            normalizeOrder();
            renderList();
            scheduleSync();
          });

          regions.on('region-clicked', (region, event) => {
            event.stopPropagation();
            playRange(region);
          });

          playButton.onclick = () => {
            stopAt = null;
            playingSplice = false;
            ws.playPause();
          };
          spliceButton.onclick = () => playSplice();
          clearButton.onclick = () => {
            playingSplice = false;
            stopAt = null;
            regions.clearRegions();
            orderMap.clear();
            selectedId = null;
            renderList();
            scheduleSync();
          };
          addButton.onclick = () => {
            if (regions.getRegions().length >= maxRegions) {
              setStatus(`Maximum $${maxRegions} regions per reference.`, true);
              return;
            }
            const duration = ws.getDuration();
            const start = Math.max(0, Math.min(ws.getCurrentTime(), Math.max(0, duration - 0.15)));
            const end = Math.min(duration, start + 3.0);
            if (end - start < 0.15) return;
            regions.addRegion({
              start, end,
              color: 'rgba(26,115,232,.22)',
              drag: true,
              resize: true,
              minLength: 0.15,
            });
          };
          zoomInput.oninput = () => ws.zoom(Number(zoomInput.value));

          root.addEventListener('keydown', (event) => {
            if (event.code === 'Space') {
              event.preventDefault();
              stopAt = null;
              playingSplice = false;
              ws.playPause();
            }
          });

        } catch (error) {
          console.error(error);
          setStatus('WaveSurfer UI could not load. Check that this Colab output is trusted and that CDN access is allowed. ' + error, true);
          document.getElementById('$audio_id').style.display = 'block';
        }
      })();
      </script>
    </div>
    """
    )


    def render_waveform_editor():
        with waveform_output:
            clear_output(wait=True)
            state["editor_token"] = None
            if not state.get("original_path"):
                display(
                    HTML(
                        "<div style='padding:18px;border:1px dashed rgba(127,127,127,.35);border-radius:14px;opacity:.62'>"
                        "Upload a reference recording to open the waveform editor.</div>"
                    )
                )
                return

            audio_url = file_to_data_url(state["original_path"], "audio/wav")
            token = uuid.uuid4().hex[:10]
            state["editor_token"] = token
            state["editor_revision"] = -1
            identifiers = {
                "root_id": f"q3root_{token}",
                "wave_id": f"q3wave_{token}",
                "list_id": f"q3list_{token}",
                "status_id": f"q3status_{token}",
                "play_id": f"q3play_{token}",
                "play_splice_id": f"q3splice_{token}",
                "add_id": f"q3add_{token}",
                "clear_id": f"q3clear_{token}",
                "zoom_id": f"q3zoom_{token}",
                "time_id": f"q3time_{token}",
                "audio_id": f"q3audio_{token}",
            }
            markup = WAVEFORM_TEMPLATE.substitute(
                **identifiers,
                callback_name=SEGMENTS_CALLBACK,
                editor_token=token,
                segments_json=json.dumps(ordered_segments()),
                max_regions=STUDIO_MAX_REGIONS,
                duration=json.dumps(float(state["duration"])),
                audio_url=audio_url,
                wavesurfer_version=WAVESURFER_VERSION,
            )
            display(HTML(markup))


    def _segments_callback(payload):
        try:
            if isinstance(payload, str):
                payload = json.loads(payload)
            if not isinstance(payload, dict):
                raise ValueError("Invalid region payload.")
            if not state.get("editor_token") or payload.get("editor_token") != state["editor_token"]:
                raise ValueError("This editor is stale. Use the most recently rendered waveform.")
            revision = int(payload.get("revision", -1))
            if revision <= state["editor_revision"]:
                raise ValueError("Out-of-order region update ignored.")
            incoming = payload.get("segments") or []
            if not isinstance(incoming, list) or len(incoming) > STUDIO_MAX_REGIONS:
                raise ValueError("Invalid number of splice regions.")
            duration = float(state.get("duration") or 0.0)
            cleaned = []
            for index, segment in enumerate(incoming[:STUDIO_MAX_REGIONS]):
                start, end = float(segment["start"]), float(segment["end"])
                if not math.isfinite(start) or not math.isfinite(end):
                    raise ValueError("Region times must be finite numbers.")
                start = max(0.0, min(start, duration))
                end = max(start, min(end, duration))
                if end - start < 0.05:
                    continue
                cleaned.append(
                    {
                        "id": str(segment.get("id") or f"segment_{index + 1}"),
                        "start": start,
                        "end": end,
                        "order": int(segment.get("order", index)),
                    }
                )
            cleaned.sort(key=lambda item: item["order"])
            new_signature = segment_signature(cleaned)
            changed = new_signature != state.get("segments_signature")
            state["editor_revision"] = revision
            state["segments"] = cleaned
            state["segments_signature"] = new_signature
            state["selected_region_id"] = payload.get("selected_id")
            if changed:
                _mark_splice_dirty()
                # A transcript must match the assembled ICL reference exactly.
                if reference_transcript.value.strip():
                    reference_transcript.value = ""
                with splice_player:
                    clear_output()
            update_segment_summary()
            return JSON({"ok": True, "count": len(cleaned)})
        except Exception as error:
            log(f"Region sync error: {error}")
            return JSON({"ok": False, "error": str(error)})


    output.register_callback(SEGMENTS_CALLBACK, _segments_callback)


    def on_reference_upload(_):
        set_busy(reference_upload, True, "Choose file…")
        try:
            set_status("Choose one audio file in the Colab file picker.", "info")
            with upload_transport:
                clear_output(wait=True)
                # Colab writes uploads to disk too; contain and remove that duplicate.
                with tempfile.TemporaryDirectory(dir=STUDIO_WORK_DIR) as upload_dir:
                    uploaded = files.upload(target_dir=upload_dir)
            if not uploaded:
                set_status("Upload cancelled.", "warning")
                return
            if len(uploaded) != 1:
                raise ValueError("Upload exactly one reference audio file.")

            uploaded_name, content = next(iter(uploaded.items()))
            name = Path(uploaded_name).name
            extension = Path(name).suffix.lower()

            if extension not in ALLOWED_AUDIO_EXTENSIONS:
                raise ValueError(
                    "Unsupported audio format. Use WAV, MP3, M4A, FLAC, OGG, AAC, "
                    "OPUS, or WEBM."
                )

            size_mib = len(content) / (1024 ** 2)
            if size_mib > STUDIO_MAX_UPLOAD_MB:
                raise ValueError(
                    f"Upload is {size_mib:.1f} MiB; the limit is "
                    f"{STUDIO_MAX_UPLOAD_MB} MiB."
                )

            set_busy(reference_upload, True, "Processing…")
            set_status("Decoding and normalizing the reference audio…", "info")
            log(f"Loading reference: {name}")

            normalized_path, duration = normalize_uploaded_audio_v3(name, content)
            if duration > STUDIO_MAX_SOURCE_SECONDS:
                Path(normalized_path).unlink(missing_ok=True)
                raise ValueError(
                    f"Source audio is {duration / 60:.1f} minutes; keep uploads at or "
                    f"below {STUDIO_MAX_SOURCE_SECONDS / 60:g} minutes."
                )

            audio, sr = load_mono_audio(normalized_path)
            metrics = audio_metrics_from_array(audio, sr)

            # Replace the previous temporary reference without touching generated outputs.
            previous_reference = state.get("original_path")
            if previous_reference and previous_reference != normalized_path:
                try:
                    previous_path = Path(previous_reference)
                    if previous_path.exists() and STUDIO_WORK_DIR in previous_path.parents:
                        previous_path.unlink(missing_ok=True)
                except Exception:
                    pass

            default_end = min(duration, 10.0, STUDIO_MAX_REFERENCE_SECONDS)
            default_segments = [
                {"id": "segment_1", "start": 0.0, "end": default_end, "order": 0}
            ] if default_end >= 0.15 else []

            state.update(
                {
                    "original_path": normalized_path,
                    "original_name": name,
                    "duration": duration,
                    "segments": default_segments,
                    "selected_region_id": "segment_1" if default_segments else None,
                    "segments_signature": segment_signature(default_segments),
                    "stitched_path": None,
                    "stitched_duration": 0.0,
                    "stitched_speech_duration": 0.0,
                }
            )
            invalidate_prompt_cache()

            reference_info.value = (
                f"<div><b>{html.escape(name)}</b></div>"
                f"<div style='opacity:.62;font-size:12px;margin-top:3px'>"
                f"{seconds_to_timestamp(duration)} · {sr/1000:.0f} kHz mono · "
                f"peak {metrics['peak']:.3f} · RMS {metrics['rms_dbfs']:.1f} dBFS</div>"
            )
            reference_transcript.value = ""
            rerender_editor_button.disabled = False
            update_segment_summary()
            render_waveform_editor()
            set_status(
                "Reference loaded. Drag across the waveform to create multiple splice regions, then reorder them below the waveform.",
                "success",
            )
            log(f"Reference normalized to {sr} Hz mono ({duration:.2f}s)")
        except Exception as error:
            set_status(f"Upload error: {html.escape(str(error))}", "error")
            log(f"Upload error: {error}")
        finally:
            with upload_transport:
                clear_output(wait=True)
            set_busy(reference_upload, False)


    reference_upload.on_click(on_reference_upload)
    rerender_editor_button.on_click(lambda _: render_waveform_editor())


    # =========================================================
    # Build / preview / export the assembled reference splice
    # =========================================================


    def prepare_spliced_reference(force=False):
        if not state.get("original_path"):
            raise ValueError("Upload reference audio first.")

        segments = ordered_segments()
        if not segments:
            raise ValueError("Create at least one waveform region first.")

        speech_duration = sum(item["end"] - item["start"] for item in segments)
        if speech_duration < STUDIO_MIN_REFERENCE_SECONDS:
            raise ValueError(
                f"The selected speech is only {speech_duration:.2f}s. "
                f"Use at least {STUDIO_MIN_REFERENCE_SECONDS:g}s."
            )
        if speech_duration > STUDIO_MAX_REFERENCE_SECONDS + 1e-6:
            raise ValueError(
                f"The splice contains {speech_duration:.2f}s of selected speech. "
                f"Keep the total at or below {STUDIO_MAX_REFERENCE_SECONDS:g}s."
            )

        signature = (
            segment_signature(segments),
            int(splice_gap_ms.value),
            int(splice_fade_ms.value),
        )
        existing_signature = state.get("stitched_signature")
        existing_path = state.get("stitched_path")
        if (
            not force
            and existing_path
            and Path(existing_path).exists()
            and existing_signature == signature
        ):
            return existing_path, state["stitched_duration"], speech_duration

        audio, sr = load_mono_audio(state["original_path"])
        gap_samples = max(0, int(round(int(splice_gap_ms.value) * sr / 1000.0)))
        fade_samples = max(0, int(round(int(splice_fade_ms.value) * sr / 1000.0)))
        gap = np.zeros(gap_samples, dtype=np.float32)
        parts = []

        for index, segment in enumerate(segments):
            start_sample = max(0, min(len(audio), int(round(segment["start"] * sr))))
            end_sample = max(start_sample, min(len(audio), int(round(segment["end"] * sr))))
            part = audio[start_sample:end_sample].astype(np.float32, copy=True)
            if part.size == 0:
                continue
            n = min(fade_samples, len(part) // 2)
            if n > 0:
                ramp = np.linspace(0.0, 1.0, n, endpoint=True, dtype=np.float32)
                part[:n] *= ramp
                part[-n:] *= ramp[::-1]
            if index and gap.size:
                parts.append(gap)
            parts.append(part)

        if not parts:
            raise ValueError("The selected regions produced no audio.")

        stitched = np.concatenate(parts).astype(np.float32, copy=False)
        peak = float(np.max(np.abs(stitched))) if stitched.size else 0.0
        if peak < 1e-5:
            raise ValueError("The assembled reference is effectively silent.")

        stitched_path = str(STUDIO_WORK_DIR / "reference_splice.wav")
        sf.write(stitched_path, stitched, sr, subtype="PCM_16")
        stitched_duration = float(len(stitched) / sr)

        state["stitched_path"] = stitched_path
        state["stitched_duration"] = stitched_duration
        state["stitched_speech_duration"] = speech_duration
        state["stitched_signature"] = signature
        return stitched_path, stitched_duration, speech_duration


    def preview_splice(_):
        set_busy(preview_splice_button, True, "Preparing…")
        try:
            path, duration, speech_duration = prepare_spliced_reference()
            with splice_player:
                clear_output(wait=True)
                display(
                    widgets.HTML(
                        f"<div style='font-size:12px;opacity:.65;margin-bottom:5px'>"
                        f"Assembled reference · {len(ordered_segments())} regions · "
                        f"{speech_duration:.2f}s speech · {duration:.2f}s including joins</div>"
                    )
                )
                display(Audio(path))
            set_status("Assembled reference splice is ready.", "success")
        except Exception as error:
            set_status(f"Splice error: {html.escape(str(error))}", "error")
        finally:
            set_busy(preview_splice_button, False)


    def download_splice(_):
        try:
            path, _, _ = prepare_spliced_reference()
            files.download(path)
        except Exception as error:
            set_status(f"Could not export splice: {html.escape(str(error))}", "error")


    preview_splice_button.on_click(preview_splice)
    download_splice_button.on_click(download_splice)


    # =========================================================
    # Clone profile + transcription
    # =========================================================

    clone_mode = widgets.ToggleButtons(
        options=[
            ("High fidelity · ICL", "icl"),
            ("Embedding only", "xvec"),
        ],
        value="icl",
        description="Mode",
        style={"description_width": "50px"},
    )
    mode_note = widgets.HTML()

    transcription_language = widgets.Dropdown(
        description="Speech",
        options=[
            ("Auto detect", None),
            ("English", "en"),
            ("Chinese", "zh"),
            ("Spanish", "es"),
            ("French", "fr"),
            ("German", "de"),
            ("Italian", "it"),
            ("Portuguese", "pt"),
            ("Japanese", "ja"),
            ("Korean", "ko"),
            ("Russian", "ru"),
        ],
        value="en",
        style={"description_width": "55px"},
        layout=widgets.Layout(width="230px"),
    )
    transcribe_button = widgets.Button(
        description="Transcribe splice",
        icon="file-text",
        button_style="warning",
        layout=widgets.Layout(width="170px"),
    )
    reference_transcript = widgets.Textarea(
        description="Transcript",
        placeholder=(
            "Transcribe the assembled splice, then correct it so it matches the selected speech exactly. "
            "The transcript should follow the splice order, not the original timeline."
        ),
        layout=widgets.Layout(width="100%", height="135px"),
        style={"description_width": "82px"},
    )
    transcript_counter = widgets.HTML()


    def update_mode_note(*_):
        invalidate_prompt_cache()
        if clone_mode.value == "icl":
            mode_note.value = (
                "<div style='font-size:12px;opacity:.68;line-height:1.45;margin:7px 0'>"
                "<b>ICL:</b> Qwen uses the assembled reference audio plus its exact transcript. "
                "This normally gives the strongest cloning fidelity.</div>"
            )
        else:
            mode_note.value = (
                "<div style='font-size:12px;opacity:.68;line-height:1.45;margin:7px 0'>"
                "<b>Embedding only:</b> transcript is not required. It is more forgiving, but voice/style fidelity can be lower.</div>"
            )


    def on_transcript_change(change=None):
        invalidate_prompt_cache()
        text = reference_transcript.value.strip()
        transcript_counter.value = (
            f"<div style='font-size:11.5px;opacity:.55;text-align:right'>{len(text):,} characters</div>"
        )


    clone_mode.observe(update_mode_note, names="value")
    reference_transcript.observe(on_transcript_change, names="value")
    update_mode_note()
    on_transcript_change()


    def transcribe_reference(_):
        set_busy(transcribe_button, True, "Transcribing…")
        try:
            path, duration, _ = prepare_spliced_reference()
            set_status(
                f"Transcribing the {seconds_to_timestamp(duration)} assembled splice with faster-whisper…",
                "info",
            )
            log(f"Whisper transcription started ({duration:.2f}s assembled reference)")

            whisper_audio = load_whisper_audio(path)
            segments, information = get_whisper_model().transcribe(
                whisper_audio,
                task="transcribe",
                language=transcription_language.value,
                beam_size=3,
                vad_filter=True,
                condition_on_previous_text=False,
                temperature=0.0,
            )
            transcript = " ".join(
                segment.text.strip() for segment in segments if segment.text.strip()
            ).strip()
            if not transcript:
                raise ValueError("No speech was detected in the assembled reference.")
            reference_transcript.value = transcript
            detected = getattr(information, "language", "unknown")
            set_status(
                f"Transcription complete · language <b>{html.escape(str(detected))}</b>. "
                "Review/correct it before ICL generation.",
                "success",
            )
            log(f"Transcription complete; language={detected}")
        except Exception as error:
            set_status(f"Transcription error: {html.escape(str(error))}", "error")
            log(f"Transcription error: {error}")
        finally:
            set_busy(transcribe_button, False)


    transcribe_button.on_click(transcribe_reference)


    # =========================================================
    # Synthesis controls
    # =========================================================

    target_text = widgets.Textarea(
        description="Script",
        placeholder="Enter what the cloned voice should say…",
        layout=widgets.Layout(width="100%", height="175px"),
        style={"description_width": "55px"},
    )
    script_counter = widgets.HTML()

    generation_language = widgets.Dropdown(
        description="Language",
        options=[
            "Auto",
            "English",
            "Chinese",
            "Japanese",
            "Korean",
            "German",
            "French",
            "Russian",
            "Portuguese",
            "Spanish",
            "Italian",
        ],
        value="English",
        style={"description_width": "68px"},
        layout=widgets.Layout(width="245px"),
    )
    output_filename = widgets.Text(
        description="File name",
        value="cloned_voice.wav",
        style={"description_width": "70px"},
        layout=widgets.Layout(width="330px"),
    )
    seed_input = widgets.IntText(
        description="Seed",
        value=-1,
        tooltip="-1 chooses a random seed each generation",
        style={"description_width": "42px"},
        layout=widgets.Layout(width="165px"),
    )

    chunk_limit = widgets.IntSlider(
        description="Chunk chars",
        value=220,
        min=80,
        max=500,
        step=10,
        continuous_update=False,
        style={"description_width": "82px"},
        layout=widgets.Layout(width="430px"),
    )
    chunk_gap = widgets.FloatSlider(
        description="Chunk gap",
        value=0.22,
        min=0.0,
        max=1.0,
        step=0.02,
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "82px"},
        layout=widgets.Layout(width="430px"),
    )
    fade_ms = widgets.IntSlider(
        description="Edge fade",
        value=8,
        min=0,
        max=40,
        step=1,
        continuous_update=False,
        style={"description_width": "82px"},
        layout=widgets.Layout(width="430px"),
    )
    peak_normalize = widgets.Checkbox(
        description="Peak-normalize final output to -1 dBFS",
        value=True,
        indent=False,
    )

    temperature = widgets.FloatSlider(
        description="Temperature",
        value=0.9,
        min=0.2,
        max=1.5,
        step=0.05,
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "92px"},
        layout=widgets.Layout(width="430px"),
    )
    top_p = widgets.FloatSlider(
        description="Top-p",
        value=1.0,
        min=0.1,
        max=1.0,
        step=0.05,
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "92px"},
        layout=widgets.Layout(width="430px"),
    )
    top_k = widgets.IntSlider(
        description="Top-k",
        value=50,
        min=1,
        max=100,
        step=1,
        continuous_update=False,
        style={"description_width": "92px"},
        layout=widgets.Layout(width="430px"),
    )
    repetition_penalty = widgets.FloatSlider(
        description="Repetition",
        value=1.05,
        min=1.0,
        max=1.5,
        step=0.01,
        readout_format=".2f",
        continuous_update=False,
        style={"description_width": "92px"},
        layout=widgets.Layout(width="430px"),
    )
    max_new_tokens = widgets.IntSlider(
        description="Max tokens",
        value=2048,
        min=256,
        max=4096,
        step=256,
        continuous_update=False,
        style={"description_width": "92px"},
        layout=widgets.Layout(width="430px"),
    )

    generate_button = widgets.Button(
        description="Generate voice",
        icon="microphone",
        button_style="success",
        layout=widgets.Layout(width="185px", height="43px"),
    )


    def update_script_counter(change=None):
        text = target_text.value
        chunks = split_text(text, chunk_limit.value) if text.strip() else []
        over = len(text) > STUDIO_MAX_SCRIPT_CHARS
        color = "#c5221f" if over else "inherit"
        script_counter.value = (
            f"<div style='font-size:11.5px;opacity:.58;text-align:right;color:{color}'>"
            f"{len(text):,}/{STUDIO_MAX_SCRIPT_CHARS:,} characters · {len(chunks)} generation section(s)</div>"
        )


    target_text.observe(update_script_counter, names="value")
    chunk_limit.observe(update_script_counter, names="value")
    update_script_counter()

    advanced_generation = widgets.VBox(
        [
            widgets.HTML(
                "<div style='font-size:12.5px;font-weight:700;margin:8px 0 7px'>Advanced generation settings</div>"
            ),
            widgets.HTML("<b>Chunk assembly</b>"),
            chunk_limit,
            chunk_gap,
            fade_ms,
            peak_normalize,
            widgets.HTML("<div style='height:9px'></div><b>Model sampling</b>"),
            temperature,
            top_p,
            top_k,
            repetition_penalty,
            max_new_tokens,
        ],
        layout=widgets.Layout(width="100%"),
    )


    # =========================================================
    # Results / interactive output waveform / history
    # =========================================================

    output_info = widgets.HTML("<span style='opacity:.62'>No generated audio yet.</span>")
    output_waveform = widgets.Output(layout=widgets.Layout(width="100%"))

    download_output = widgets.Button(
        description="Download WAV", icon="download", button_style="primary", disabled=True
    )
    download_all = widgets.Button(
        description="Download all (.zip)", icon="archive", disabled=True
    )
    clear_history = widgets.Button(
        description="Clear history", icon="trash", disabled=True
    )
    history_select = widgets.Select(
        options=[],
        description="History",
        rows=5,
        style={"description_width": "55px"},
        layout=widgets.Layout(width="100%"),
    )


    # Optional exports are separate copies. The native-rate PCM_16 master is untouched.
    def export_wav(source, destination, sample_rate=8000, channels=1, bits=16):
        source, destination = Path(source), Path(destination)
        if source.resolve() == destination.resolve():
            raise ValueError("Choose a separate export path to preserve the original.")
        if destination.exists():
            raise ValueError("The export destination already exists.")
        if sample_rate not in (8000, 16000, 22050, 24000, 44100, 48000):
            raise ValueError("Unsupported sample rate.")
        if channels not in (1, 2) or bits not in (16, 24, 32):
            raise ValueError("Choose mono/stereo and 16, 24, or 32-bit PCM.")
        command = [
            ensure_ffmpeg(), "-hide_banner", "-loglevel", "error", "-nostdin", "-n",
            "-i", str(source), "-vn", "-ac", str(channels), "-ar", str(sample_rate),
            "-c:a", f"pcm_s{bits}le", "-f", "wav", str(destination),
        ]
        try:
            subprocess.run(command, check=True, capture_output=True, text=True, timeout=120)
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
            destination.unlink(missing_ok=True)
            detail = str(getattr(error, "stderr", "") or error)[-1200:]
            raise RuntimeError(f"WAV conversion failed: {detail}") from error
        return str(destination)


    export_enabled = widgets.Checkbox(value=False, description="Enable optional WAV conversion", indent=False)
    export_rate = widgets.Dropdown(
        options=[("8 kHz", 8000), ("16 kHz", 16000), ("22.05 kHz", 22050),
                 ("24 kHz", 24000), ("44.1 kHz", 44100), ("48 kHz", 48000)],
        value=8000, description="Sample rate", style={"description_width": "90px"},
    )
    export_channels = widgets.Dropdown(
        options=[("Mono", 1), ("Stereo (duplicated mono)", 2)], value=1, description="Channels",
        style={"description_width": "90px"},
    )
    export_bits = widgets.Dropdown(
        options=[("16-bit PCM", 16), ("24-bit PCM", 24), ("32-bit PCM", 32)], value=16,
        description="Bit depth", style={"description_width": "90px"},
    )
    export_note = widgets.HTML()
    export_button = widgets.Button(description="Create WAV copy", icon="file-audio-o", disabled=True)
    export_download = widgets.Button(description="Download converted WAV", icon="download", disabled=True)
    export_preview = widgets.Output()
    export_state = {"path": None}


    def invalidate_export(*_):
        # A new history selection or settings change must not download a stale conversion.
        export_state["path"] = None
        export_download.disabled = True
        enabled = bool(export_enabled.value)
        for control in (export_rate, export_channels, export_bits):
            control.disabled = not enabled
        export_button.disabled = not (enabled and state.get("generated_path"))
        with export_preview:
            clear_output()
        bitrate = export_rate.value * export_channels.value * export_bits.value // 1000
        export_note.value = (
            "<div style='font-size:12px;opacity:.68'>Format: WAV · "
            f"Uncompressed audio bit rate: {bitrate} kb/s.<br>"
            "8 kHz is a sample rate. Conversion applies only when you click Create WAV copy. "
            "The generated master and Download WAV keep the original quality.</div>"
        )


    def create_export(_):
        set_busy(export_button, True, "Converting…")
        try:
            source = state.get("generated_path")
            if not export_enabled.value or not source or not Path(source).is_file():
                raise ValueError("Select a generated take and enable optional conversion first.")
            channel_name = "mono" if export_channels.value == 1 else "stereo"
            name = f"{Path(source).stem}_{export_rate.value}Hz_{channel_name}_{export_bits.value}bit.wav"
            destination = next_available_path(STUDIO_OUTPUT_DIR, name)
            path = export_wav(source, destination, export_rate.value, export_channels.value, export_bits.value)
            export_state["path"] = path
            export_download.disabled = False
            with export_preview:
                clear_output(wait=True)
                info = sf.info(path)
                print(f"{Path(path).name} · {info.samplerate} Hz · {info.channels} channel(s) · {info.subtype}")
                display(Audio(filename=path))
            set_status("Converted WAV ready. Use Download converted WAV to save it.", "success")
        except Exception as error:
            set_status(f"Export error: {html.escape(str(error))}", "error")
            log(f"Export error: {error}")
        finally:
            set_busy(export_button, False)
            export_button.disabled = not (export_enabled.value and state.get("generated_path"))


    def download_export(_):
        path = export_state["path"]
        if path and Path(path).is_file():
            files.download(path)
        else:
            set_status("Create a WAV copy first.", "warning")


    for control in (export_enabled, export_rate, export_channels, export_bits):
        control.observe(invalidate_export, names="value")
    export_button.on_click(create_export)
    export_download.on_click(download_export)
    invalidate_export()
    export_panel = widgets.VBox([
        widgets.HTML("<b>Optional output conversion</b>"), export_enabled,
        export_rate, export_channels, export_bits, export_note,
        widgets.HBox([export_button, export_download], layout=widgets.Layout(flex_flow="row wrap", gap="8px")),
        export_preview,
    ])


    OUTPUT_PLAYER_TEMPLATE = Template(
        r"""
    <div id="$root_id" style="border:1px solid rgba(127,127,127,.18);border-radius:15px;padding:12px;background:rgba(127,127,127,.025)">
      <div style="display:flex;gap:8px;align-items:center;margin-bottom:7px;flex-wrap:wrap">
        <button id="$play_id" style="border:1px solid rgba(26,115,232,.25);background:rgba(26,115,232,.10);color:#1967d2;border-radius:9px;padding:6px 10px;font-weight:700;cursor:pointer">▶ Play</button>
        <span id="$time_id" style="font:600 11.5px ui-monospace,SFMono-Regular,Menlo,monospace;opacity:.62">00:00.0 / 00:00.0</span>
      </div>
      <div id="$wave_id"></div>
      <audio id="$audio_id" controls src="$audio_url" style="width:100%;margin-top:8px;display:none"></audio>
      <div id="$status_id" style="font-size:11px;opacity:.55;margin-top:5px"></div>
      <script type="module">
        (async () => {
          const status = document.getElementById('$status_id');
          const button = document.getElementById('$play_id');
          const time = document.getElementById('$time_id');
          const fmt = (seconds) => {
            seconds = Math.max(0, Number(seconds || 0));
            const m = Math.floor(seconds / 60); const s = seconds - m*60;
            return `$${String(m).padStart(2,'0')}:$${s.toFixed(1).padStart(4,'0')}`;
          };
          try {
            const {default: WaveSurfer} = await import('https://cdn.jsdelivr.net/npm/wavesurfer.js@$wavesurfer_version/dist/wavesurfer.esm.js');
            const ws = WaveSurfer.create({
              container: '#$wave_id', url: '$audio_url', height: 125,
              waveColor:'#9aa0a6', progressColor:'#1a73e8', cursorColor:'#6f5cff',
              barWidth:2, barGap:1, barRadius:2, normalize:true,
            });
            ws.on('ready', d => { time.textContent = `$${fmt(0)} / $${fmt(d)}`; });
            ws.on('timeupdate', t => { time.textContent = `$${fmt(t)} / $${fmt(ws.getDuration())}`; });
            ws.on('play', () => { button.textContent = '❚❚ Pause'; });
            ws.on('pause', () => { button.textContent = '▶ Play'; });
            ws.on('click', () => ws.playPause());
            button.onclick = () => ws.playPause();
          } catch (error) {
            status.textContent = 'Interactive waveform unavailable; using native audio player.';
            document.getElementById('$audio_id').style.display = 'block';
          }
        })();
      </script>
    </div>
    """
    )


    def render_output_player(path):
        with output_waveform:
            clear_output(wait=True)
            audio_url = file_to_data_url(path, "audio/wav")
            token = uuid.uuid4().hex[:10]
            display(
                HTML(
                    OUTPUT_PLAYER_TEMPLATE.substitute(
                        root_id=f"q3outroot_{token}",
                        wave_id=f"q3outwave_{token}",
                        play_id=f"q3outplay_{token}",
                        time_id=f"q3outtime_{token}",
                        audio_id=f"q3outaudio_{token}",
                        status_id=f"q3outstatus_{token}",
                        audio_url=audio_url,
                        wavesurfer_version=WAVESURFER_VERSION,
                    )
                )
            )


    def display_history_item(item):
        path = item["path"]
        if not Path(path).exists():
            set_status("That history file no longer exists in this runtime.", "warning")
            return

        state["generated_path"] = path
        state["generated_sr"] = item["sample_rate"]
        invalidate_export()
        audio, sr = load_mono_audio(path)
        metrics = audio_metrics_from_array(audio, sr)
        output_info.value = (
            f"<div><b>{html.escape(Path(path).name)}</b></div>"
            f"<div style='opacity:.62;font-size:12px;margin-top:3px'>"
            f"{seconds_to_timestamp(metrics['duration'])} · {sr/1000:.1f} kHz · "
            f"peak {metrics['peak']:.3f} · RMS {metrics['rms_dbfs']:.1f} dBFS · seed {item['seed']}</div>"
        )
        render_output_player(path)
        download_output.disabled = False


    def refresh_history_widget(select_latest=True):
        options = []
        for index, item in enumerate(state["history"]):
            label = (
                f"{index + 1:02d} · {Path(item['path']).name} · "
                f"{item['duration']:.1f}s · seed {item['seed']}"
            )
            options.append((label, index))
        history_select.options = options
        has_history = bool(options)
        download_all.disabled = not has_history
        clear_history.disabled = not has_history
        if has_history and select_latest:
            history_select.value = len(options) - 1


    def on_history_select(change):
        if change.get("name") != "value" or change.get("new") is None:
            return
        index = int(change["new"])
        if 0 <= index < len(state["history"]):
            display_history_item(state["history"][index])


    history_select.observe(on_history_select, names="value")


    def download_current_output(_):
        path = state.get("generated_path")
        if path and Path(path).exists():
            files.download(path)
        else:
            set_status("No generated WAV is available to download.", "warning")


    def download_history_zip(_):
        valid = [item for item in state["history"] if Path(item["path"]).exists()]
        if not valid:
            set_status("No history files are available.", "warning")
            return
        zip_path = str(STUDIO_OUTPUT_DIR / "qwen3_tts_outputs.zip")
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for item in valid:
                archive.write(item["path"], arcname=Path(item["path"]).name)
        files.download(zip_path)


    def clear_output_history(_):
        state["history"] = []
        state["generated_path"] = None
        state["generated_sr"] = None
        invalidate_export()
        history_select.options = []
        output_info.value = "<span style='opacity:.62'>No generated audio yet.</span>"
        with output_waveform:
            clear_output()
        download_output.disabled = True
        download_all.disabled = True
        clear_history.disabled = True
        set_status(
            f"Generation history cleared from the panel. Existing files remain in "
            f"{html.escape(str(STUDIO_OUTPUT_DIR))}.",
            "info",
        )


    download_output.on_click(download_current_output)
    download_all.on_click(download_history_zip)
    clear_history.on_click(clear_output_history)


    # =========================================================
    # Voice generation
    # =========================================================


    def generate_voice(_):
        set_busy(generate_button, True, "Generating…")
        progress.layout.display = "flex"
        progress.value = 0

        try:
            if not state.get("original_path"):
                raise ValueError("Upload reference audio first.")

            script = target_text.value.strip()
            transcript = reference_transcript.value.strip()
            if not script:
                raise ValueError("Enter a script to synthesize.")
            if len(script) > STUDIO_MAX_SCRIPT_CHARS:
                raise ValueError(
                    f"Script is {len(script):,} characters; keep each run under {STUDIO_MAX_SCRIPT_CHARS:,}."
                )
            if clone_mode.value == "icl" and not transcript:
                raise ValueError(
                    "High-fidelity ICL mode requires the exact transcript of the assembled splice. "
                    "Use Transcribe splice, then correct it if needed."
                )

            requested_seed = int(seed_input.value)
            if not -1 <= requested_seed <= 2**63 - 1:
                raise ValueError("Seed must be -1 (random) or an integer from 0 to 2^63 - 1.")
            splice_path, splice_duration, speech_duration = prepare_spliced_reference()
            xvec_only = clone_mode.value == "xvec"
            splice_sig = state.get("stitched_signature")
            prompt_key = (
                splice_sig,
                xvec_only,
                None if xvec_only else transcript,
            )

            chunks = split_text(script, chunk_limit.value)
            if not chunks:
                raise ValueError("The script is empty after text processing.")

            tts_model = get_tts_model()
            progress.max = len(chunks) + 1
            progress.description = "Prompt"
            set_status(
                f"Preparing clone prompt from {len(ordered_segments())} splice regions ({speech_duration:.2f}s speech)…",
                "info",
            )

            if state["voice_prompt"] is None or state["voice_prompt_key"] != prompt_key:
                with torch.inference_mode():
                    state["voice_prompt"] = tts_model.create_voice_clone_prompt(
                        ref_audio=load_mono_audio(splice_path),
                        ref_text=None if xvec_only else transcript,
                        x_vector_only_mode=xvec_only,
                    )
                state["voice_prompt_key"] = prompt_key
                log(
                    f"Created voice prompt from splice ({'speaker embedding only' if xvec_only else 'ICL'})"
                )
            else:
                log("Reused cached voice prompt")

            progress.value = 1
            actual_seed = (
                requested_seed
                if requested_seed >= 0
                else random.SystemRandom().randrange(0, 2**31 - 1)
            )
            random.seed(actual_seed)
            np.random.seed(actual_seed % (2**32 - 1))
            torch.manual_seed(actual_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(actual_seed)

            log(f"Generation seed={actual_seed}; sections={len(chunks)}")
            generated_parts = []
            output_sr = None

            for index, chunk in enumerate(chunks, start=1):
                progress.description = f"{index}/{len(chunks)}"
                set_status(
                    f"Generating section <b>{index}/{len(chunks)}</b> · {len(chunk)} characters…",
                    "info",
                )
                with torch.inference_mode():
                    wavs, sr = tts_model.generate_voice_clone(
                        text=chunk,
                        language=generation_language.value,
                        voice_clone_prompt=state["voice_prompt"],
                        max_new_tokens=int(max_new_tokens.value),
                        do_sample=True,
                        top_k=int(top_k.value),
                        top_p=float(top_p.value),
                        temperature=float(temperature.value),
                        repetition_penalty=float(repetition_penalty.value),
                        subtalker_dosample=True,
                        subtalker_top_k=int(top_k.value),
                        subtalker_top_p=float(top_p.value),
                        subtalker_temperature=float(temperature.value),
                    )

                if not wavs:
                    raise RuntimeError("The model returned no audio.")
                if int(sr) <= 0:
                    raise RuntimeError("The model returned an invalid sample rate.")
                if output_sr is None:
                    output_sr = int(sr)
                elif int(sr) != output_sr:
                    raise RuntimeError("The model returned inconsistent sample rates across chunks.")
                generated_parts.append(sanitize_generated_audio(wavs[0]))
                progress.value = index + 1

            final_audio = join_generated_audio(
                generated_parts,
                output_sr,
                gap_seconds=float(chunk_gap.value),
                fade_ms=int(fade_ms.value),
                peak_normalize=bool(peak_normalize.value),
            )

            if np.max(np.abs(final_audio)) < 1e-5:
                raise RuntimeError("The model returned silent audio. No output was saved; try a cleaner reference.")
            final_path = next_available_path(
                STUDIO_OUTPUT_DIR, str(Path(_safe_filename(output_filename.value, "cloned_voice.wav")).with_suffix(".wav"))
            )
            sf.write(final_path, final_audio, output_sr, format="WAV", subtype="PCM_16")
            metrics = audio_metrics_from_array(final_audio, output_sr)
            item = {
                "path": final_path,
                "sample_rate": output_sr,
                "duration": metrics["duration"],
                "seed": actual_seed,
                "created": datetime.now().isoformat(timespec="seconds"),
                "mode": clone_mode.value,
                "script": script,
                "reference_regions": len(ordered_segments()),
                "reference_speech_seconds": speech_duration,
                "reference_assembled_seconds": splice_duration,
            }
            state["history"].append(item)
            state["generated_path"] = final_path
            state["generated_sr"] = output_sr

            refresh_history_widget(select_latest=True)
            set_status(
                f"Generation complete · {metrics['duration']:.1f}s · seed <b>{actual_seed}</b>.",
                "success",
            )
            log(f"Saved {final_path} ({metrics['duration']:.2f}s @ {output_sr} Hz)")

        except torch.cuda.OutOfMemoryError:
            invalidate_prompt_cache()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            set_status(
                "GPU out of memory. Reconnect the runtime and use MODEL_SIZE='0.6B', shorten the script/chunks, "
                "or use a GPU with more VRAM.",
                "error",
            )
            log("CUDA out-of-memory during generation")
        except Exception as error:
            set_status(f"Generation error: {html.escape(str(error))}", "error")
            log(f"Generation error: {error}")
        finally:
            progress.description = ""
            progress.layout.display = "none"
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            set_busy(generate_button, False)


    generate_button.on_click(generate_voice)


    # =========================================================
    # Maintenance
    # =========================================================

    clear_gpu_button = widgets.Button(description="Clear GPU cache", icon="eraser")
    reset_panel_button = widgets.Button(description="Reset studio", icon="refresh")


    def clear_gpu_cache(_):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        set_status("Python garbage collection and CUDA cache cleanup completed.", "info")
        log("GPU cache cleared")


    def reset_panel(_):
        # Clean temporary reference/splice files while preserving generated outputs.
        for key in ("original_path", "stitched_path"):
            candidate = state.get(key)
            if candidate:
                try:
                    path = Path(candidate)
                    if path.exists() and STUDIO_WORK_DIR in path.parents:
                        path.unlink(missing_ok=True)
                except Exception:
                    pass

        state.update(
            {
                "editor_token": None,
                "editor_revision": -1,
                "original_path": None,
                "original_name": None,
                "duration": 0.0,
                "segments": [],
                "selected_region_id": None,
                "segments_signature": None,
                "stitched_path": None,
                "stitched_duration": 0.0,
                "stitched_speech_duration": 0.0,
                "stitched_signature": None,
                "voice_prompt": None,
                "voice_prompt_key": None,
                "generated_path": None,
                "generated_sr": None,
                "history": [],
            }
        )
        reference_info.value = "<span style='opacity:.62'>No reference audio loaded.</span>"
        reference_transcript.value = ""
        target_text.value = ""
        clone_mode.value = "icl"
        generation_language.value = "English"
        transcription_language.value = "en"
        output_filename.value = "cloned_voice.wav"
        seed_input.value = -1
        export_enabled.value = False
        invalidate_export()
        history_select.options = []
        output_info.value = "<span style='opacity:.62'>No generated audio yet.</span>"
        rerender_editor_button.disabled = True
        preview_splice_button.disabled = True
        download_splice_button.disabled = True
        download_output.disabled = True
        download_all.disabled = True
        clear_history.disabled = True
        with waveform_output:
            clear_output()
        with splice_player:
            clear_output()
        with output_waveform:
            clear_output()
        with log_output:
            clear_output()
        render_waveform_editor()
        update_segment_summary()
        update_mode_note()
        update_script_counter()
        set_status("Studio reset. Runtime models remain loaded.", "info")


    clear_gpu_button.on_click(clear_gpu_cache)
    reset_panel_button.on_click(reset_panel)


    # =========================================================
    # Assemble Colab-safe production interface
    # =========================================================

    reference_card = card(
        section_heading(
            "◉",
            "Reference & splice editor",
            "Upload once, then build a clean reference from multiple non-contiguous sections directly on the waveform.",
        ),
        widgets.HBox(
            [reference_upload, reference_info],
            layout=widgets.Layout(
                width="100%", gap="14px", align_items="center", flex_flow="row wrap"
            ),
        ),
        upload_transport,
        widgets.HTML("<div style='height:10px'></div>"),
        waveform_output,
        segment_summary,
        widgets.HBox(
            [preview_splice_button, download_splice_button, rerender_editor_button],
            layout=widgets.Layout(
                width="100%", gap="8px", align_items="center", flex_flow="row wrap", margin="10px 0 4px"
            ),
        ),
        splice_player,
        editor_settings,
    )

    profile_card = card(
        section_heading(
            "Aa",
            "Clone profile",
            "Choose the clone method and prepare the transcript for the assembled splice.",
        ),
        clone_mode,
        mode_note,
        widgets.HBox(
            [transcription_language, transcribe_button],
            layout=widgets.Layout(
                width="100%", gap="9px", align_items="center", flex_flow="row wrap", margin="8px 0 10px"
            ),
        ),
        reference_transcript,
        transcript_counter,
    )

    advanced_toggle = widgets.ToggleButton(description="Advanced settings", icon="sliders", value=False)
    advanced_generation.layout.display = "none"
    def toggle_advanced(change):
        advanced_generation.layout.display = "flex" if change["new"] else "none"
    advanced_toggle.observe(toggle_advanced, names="value")

    generation_card = card(
        section_heading(
            "✦",
            "Synthesis",
            "Generate new speech with the assembled clone reference. Long scripts are chunked and joined automatically.",
        ),
        target_text,
        script_counter,
        widgets.HBox(
            [generation_language, output_filename, seed_input],
            layout=widgets.Layout(
                width="100%", gap="10px", align_items="center", flex_flow="row wrap", margin="10px 0"
            ),
        ),
        advanced_toggle,
        advanced_generation,
        widgets.Box([generate_button], layout=widgets.Layout(margin="12px 0 3px")),
        progress,
    )

    results_card = card(
        section_heading(
            "▶",
            "Output",
            "Play from the generated waveform, revisit earlier takes, or export WAV files.",
        ),
        output_info,
        widgets.HTML("<div style='height:7px'></div>"),
        output_waveform,
        widgets.HBox(
            [download_output, download_all, clear_history],
            layout=widgets.Layout(
                width="100%", gap="8px", flex_flow="row wrap", margin="10px 0"
            ),
        ),
        history_select,
        export_panel,
    )

    workspace_steps = widgets.HTML(
        """
        <div style="display:flex;gap:8px;flex-wrap:wrap;margin:2px 0 12px">
          <span style="padding:7px 10px;border-radius:999px;background:rgba(26,115,232,.11);font-size:11.5px;font-weight:700;color:#1967d2">1 · Reference</span>
          <span style="padding:7px 10px;border-radius:999px;background:rgba(127,127,127,.08);font-size:11.5px;font-weight:700">2 · Clone profile</span>
          <span style="padding:7px 10px;border-radius:999px;background:rgba(127,127,127,.08);font-size:11.5px;font-weight:700">3 · Synthesis</span>
          <span style="padding:7px 10px;border-radius:999px;background:rgba(127,127,127,.08);font-size:11.5px;font-weight:700">4 · Output</span>
        </div>
        """
    )

    logs_card = card(
        section_heading(
            "⌁",
            "Runtime log",
            "Useful for diagnosing upload, transcription, waveform, or generation errors.",
        ),
        log_output,
    )

    usage_note = widgets.HTML(
        """
        <div style="font-size:11.5px;opacity:.58;line-height:1.45;margin:6px 2px 2px">
          Reference audio and generated files stay inside the active Colab runtime
          unless you explicitly download them. Use only voices and recordings you
          have permission to clone.
        </div>
        """
    )

    footer = widgets.HBox(
        [clear_gpu_button, reset_panel_button],
        layout=widgets.Layout(
            width="100%", gap="8px", justify_content="flex-end", flex_flow="row wrap", margin="8px 0"
        ),
    )

    # Colab-safe layout: deliberately avoid widgets.Tab / widgets.Accordion.
    # Some Colab front-end/widget-manager combinations can collapse selection containers
    # while ordinary VBox/HBox/Button widgets still render. Keeping the four stages as
    # normal stacked cards makes the upload control reliably visible.
    interface = widgets.VBox(
        [
            STYLE,
            header,
            model_status,
            status_banner,
            workspace_steps,
            reference_card,
            profile_card,
            generation_card,
            results_card,
            logs_card,
            usage_note,
            footer,
        ],
        layout=widgets.Layout(width="100%", max_width="1040px", margin="0 auto"),
    )
    interface.add_class("q3v3-shell")

    display(interface)
    render_waveform_editor()
    update_segment_summary()
    set_status(
        "<b>READY — both models are loaded.</b> You can start voice cloning: upload your reference, transcribe it, and generate.",
        "success",
    )


    def is_ready():
        return all(models.get(name) is not None for name in ("tts", "whisper"))


    def redisplay():
        if not is_ready():
            raise RuntimeError("Models are no longer loaded. Rerun the launch cell.")
        display(interface)
        set_status("<b>READY — both models are already loaded.</b> You can continue voice cloning.", "success")
        render_waveform_editor()
        if state.get("generated_path") and Path(state["generated_path"]).is_file():
            render_output_player(state["generated_path"])
        if export_state["path"] and Path(export_state["path"]).is_file():
            with export_preview:
                clear_output(wait=True)
                display(Audio(filename=export_state["path"]))


    def close_studio():
        models.clear()
        invalidate_prompt_cache()
        # Replace the old callback so a retired editor cannot retain the old studio.
        output.register_callback(SEGMENTS_CALLBACK, lambda *_: {"ok": False, "error": "Studio was closed; use the current editor."})
        def close_widget(widget):
            for child in getattr(widget, "children", ()):
                close_widget(child)
            widget.close()
        close_widget(interface)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    def self_check_audio():
        """Optional model-free regression check: studio['self_check_audio']()."""
        import tempfile
        import wave
        import hashlib
        with tempfile.TemporaryDirectory() as directory:
            source = Path(directory) / "master.wav"
            converted = Path(directory) / "phone.wav"
            sr = 24000
            signal = (0.25 * np.sin(2 * np.pi * 440 * np.arange(sr) / sr)).astype(np.float32)
            sf.write(source, signal, sr, subtype="PCM_16")
            whisper_audio = load_whisper_audio(source)
            assert whisper_audio.shape == (16000,)
            assert whisper_audio.dtype == np.float32 and whisper_audio.flags.c_contiguous
            # Resampling must preserve pitch, not reinterpret 24 kHz samples as 16 kHz.
            peak_hz = np.argmax(np.abs(np.fft.rfft(whisper_audio)))
            assert abs(peak_hz - 440) <= 1
            for invalid in (np.array([np.nan]), np.array([np.inf]), np.array([])):
                try:
                    sanitize_generated_audio(invalid)
                except ValueError:
                    pass
                else:
                    raise AssertionError("Invalid generated audio was accepted")
            before = hashlib.sha256(source.read_bytes()).digest()
            export_wav(source, converted, 8000, 1, 16)
            with wave.open(str(converted), "rb") as audio:
                assert (audio.getframerate(), audio.getnchannels(), audio.getsampwidth()) == (8000, 1, 2)
                assert abs(audio.getnframes() / 8000 - 1.0) < 0.01
            assert hashlib.sha256(source.read_bytes()).digest() == before
            chunks = split_text("Long sentence. " * 60, 80)
            assert chunks and all(len(chunk) <= 80 for chunk in chunks)
            joined = join_generated_audio([signal, signal], sr, gap_seconds=0.22)
            assert len(joined) == 2 * sr + round(0.22 * sr)
        print("Audio checks passed: Whisper 16 kHz/pitch, invalid model audio, WAV headers, master preservation, chunking and joins.")


    update_model_status()
    return {"version": STUDIO_VERSION, "config": (model_size, whisper_size), "redisplay": redisplay, "close": close_studio,
            "self_check_audio": self_check_audio, "is_ready": is_ready}


def start_studio(model_size, whisper_size):
    models = load_studio_models(model_size, whisper_size)
    try:
        print("[4/4] Opening the studio…", flush=True)
        studio = create_studio(model_size, whisper_size, models)
    except Exception as error:
        models.clear()
        import gc
        import torch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"STARTUP FAILED — could not open the studio: {error}", flush=True)
        raise
    print("✅ READY — both models are loaded. You can start voice cloning now.", flush=True)
    return studio


# Successful reruns reuse loaded models and current edits; downloads use the HF cache.
if ("studio" in globals() and studio.get("version") == STUDIO_VERSION
        and studio["config"] == (MODEL_SIZE, WHISPER_SIZE)
        and studio.get("is_ready", lambda: False)()):
    studio["redisplay"]()
    print("✅ READY — both models are already loaded. You can continue voice cloning.", flush=True)
else:
    if "studio" in globals():
        studio["close"]()
        del studio
    studio = start_studio(MODEL_SIZE, WHISPER_SIZE)


## Advanced generation settings

### Joining chunks

Long scripts are split into smaller pieces (chunks), generated one at a time, then joined.

**Chunk chars (default `220`, range `180–300`)**
How many characters go into each chunk. Lower uses less GPU memory and is more stable. Higher means fewer joins, but long chunks can drift in quality.

**Chunk gap (default `0.22` s, range `0.15–0.30`)**
The silence added between chunks. Raise it if sentences run together; lower it if pauses feel too long.

**Edge fade (default `8 ms`, range `5–15 ms`)**
A tiny fade at the start and end of each chunk to prevent clicks at the joins. Keep it small.

**Normalize to `-1 dBFS` (recommended: ON)**
Sets the final volume so the loudest point sits just below the maximum, without clipping. It evens out volume but doesn't squash the sound.

---

### Sampling

**Temperature (default `0.90`, range `0.70–1.00`)**
How much variation the model uses. Lower is more consistent; higher is more expressive but less stable. For a closer match to the cloned voice, try `0.75–0.85`.

**Top-p (default `1.00`, range `0.85–1.00`)**
Limits the model to its most likely choices. `1.00` means no limit; lower values make output more cautious.

**Top-k (default `50`)**
Limits each step to the K most likely choices. Lower is more predictable; higher adds variety. `50` suits most uses.

**Repetition (default `1.05`, range `1.03–1.10`)**
Discourages repeated words, sounds, or phrases. `1.00` turns it off. Setting it too high can make speech sound unnatural.

**Max tokens (default `2048`)**
The maximum length of audio generated per chunk. If it's too low, speech may cut off early; higher allows longer output but uses more time and memory. `2048` works well with 220-character chunks.